### Final-Year Project — Charlotte Droin & Bertrand Lecoeur

# Project 2: Pricing and Calibration of VIX Options

The objective is to study the pricing and calibration of VIX options within a stochastic forward variance curve model.


## 1 Initialisation and Mathematical Functions

## 1.1 Initialisation and Pricing Functions
Import of the modules required for the project and definition of the basic analytical functions. The market convention for evaluating the implied volatility of VIX options is to use the VIX future as the underlying input in the Black formula.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq, least_squares
from google.colab import files

# Import local files
uploaded = files.upload()

# Define analytical pricing functions
def black_call(F, K, T, vol):
    # Return intrinsic value if volatility or maturity is zero
    if vol <= 0 or T <= 0: return max(F - K, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return F * norm.cdf(d1) - K * norm.cdf(d2)

def black_put(F, K, T, vol):
    # Return intrinsic value if volatility or maturity is zero
    if vol <= 0 or T <= 0: return max(K - F, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return K * norm.cdf(-d2) - F * norm.cdf(-d1)

# Numerical inversion to extract implied volatility
def implied_vol_black(price, F, K, T, is_call=True):
    intrinsic = max(F - K, 0.0) if is_call else max(K - F, 0.0)
    if price <= intrinsic + 1e-12:
        return np.nan
    try:
        if is_call:
            return brentq(lambda vol: black_call(F, K, T, vol) - price, 1e-6, 10.0)
        else:
            return brentq(lambda vol: black_put(F, K, T, vol) - price, 1e-6, 10.0)
    except:
        return np.nan


## 1.2 Market Data Processing
The data are loaded and the target maturity is selected. We restrict the sample to quotes with positive prices. The reference price used is $Mid=\frac{1}{2}(Bid+Ask)$. The implied VIX future ($F_{0}$) is then derived from call-put parity.

In [ ]:
# Read the quote file
df = pd.read_csv("vix_quotedata (2).csv", skiprows=3)

# Filter for the target maturity
df = df[df["Expiration Date"] == "Wed Apr 29 2026"].copy()

# Convert price columns to numeric format
cols_to_numeric = ["Strike", "Bid", "Ask", "Bid.1", "Ask.1"]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Exclude rows without strictly positive quotes
df = df[(df["Bid"] > 0) & (df["Ask"] > 0) & (df["Bid.1"] > 0) & (df["Ask.1"] > 0)].copy()

# Calculate Mid prices for Calls and Puts
df["Mid_Call"] = 0.5 * (df["Bid"] + df["Ask"])
df["Mid_Put"] = 0.5 * (df["Bid.1"] + df["Ask.1"])

# Calculate the VIX Future using Call-Put parity
df["Synthetic_Forward"] = df["Mid_Call"] - df["Mid_Put"] + df["Strike"]
F0_mkt = df["Synthetic_Forward"].median()
print(f"Implied VIX Future derived from the market (F0) : {F0_mkt:.4f}\n")

## 1.3 Implied Volatility Extraction and Smile Visualisation
Using the calculated $Mid$ prices and the futures value, implied volatility is extracted numerically. The data are then restricted to the relevant strike range in order to visualise the volatility smile.

In [ ]:
# Initialise maturity and result lists
T_mkt = 16 / 365
iv_recalc_calls = []
iv_recalc_puts = []

# Iteratively calculate implied volatility for each strike
for index, row in df.iterrows():
    K = row["Strike"]
    iv_recalc_calls.append(implied_vol_black(row["Mid_Call"], F0_mkt, K, T_mkt, is_call=True))
    iv_recalc_puts.append(implied_vol_black(row["Mid_Put"], F0_mkt, K, T_mkt, is_call=False))

# Add results to the DataFrame
df["IV_Recalc_Call"] = iv_recalc_calls
df["IV_Recalc_Put"] = iv_recalc_puts

# Remove undefined values and restrict the strike range
df_clean = df.dropna(subset=["IV_Recalc_Call", "IV_Recalc_Put"]).copy()
df_clean = df_clean[(df_clean["Strike"] >= 15) & (df_clean["Strike"] <= 25)]

# Extract final vectors for visualisation and subsequent analysis
strikes_mkt = df_clean["Strike"].values
iv_mkt_calls = df_clean["IV_Recalc_Call"].values
iv_mkt_puts = df_clean["IV_Recalc_Put"].values

# Build the volatility smile chart
plt.figure(figsize=(9,5))
plt.plot(strikes_mkt, iv_mkt_calls, "o-", color="blue", lw=2, label="Market - Calls")
plt.plot(strikes_mkt, iv_mkt_puts, "x--", color="blue", lw=2, label="Market - Puts")
plt.axvline(F0_mkt, color="gray", ls=":", label=f"Future VIX ≈ {F0_mkt:.2f}")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("Market VIX Smile - Calls vs Puts")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 2. The One-Factor Bergomi Model

In this model, the function $\varphi$ is defined by a decreasing exponential:
$$\varphi(\tau) = \omega e^{-k\tau}$$
where $\omega$ represents volatility of variance and $1/k$ the characteristic time scale.


## 2.1 One-Factor Bergomi Model Simulation
Here, the standard Bergomi model is implemented. The forward variance integral is approximated using the rectangle rule. The model's exponential specification allows a Markovian representation using the Ornstein--Uhlenbeck process $X_{T}$, simulated by Monte Carlo together with antithetic variates to reduce variance.

In [ ]:
# Define model parameters
r = 0.0                # Interest rate set by assumption
delta = 1/12           # Approximately 30 days
T = T_mkt              # Valuation maturity
sigma0 = 0.20          # Initial volatility
xi0 = sigma0**2        # Initial variance
omega = 1.5            # Volatility of variance
k = 3.0                # Mean-reversion parameter

# Monte Carlo simulation setup
n_mc = 50000
n_rect = 120
np.random.seed(42)

# Simulate the forward variance factor at maturity
# Ornstein-Uhlenbeck process for X_T
var_XT = (1.0 - np.exp(-2.0 * k * T)) / (2.0 * k)
Z = np.random.randn(n_mc // 2)
Z_anti = np.concatenate((Z, -Z))
X_T = np.sqrt(var_XT) * Z_anti

# Discretise the integral using the rectangle rule
du = delta / n_rect
vix2 = np.zeros(n_mc)

for j in range(n_rect):
    u = T + (j + 0.5) * du
    a = np.exp(-k * (u - T))
    xi_T_u = xi0 * np.exp(omega * a * X_T - 0.5 * (omega**2) * (a**2) * var_XT)
    vix2 += xi_T_u * du

vix2 = vix2 / delta
VIX_T_sim = np.sqrt(vix2)

# Calculate the expectation to obtain the simulated VIX Future
F0_model = np.mean(VIX_T_sim)

## 2.2 Pricing and Implied Volatility Extraction
Using the simulated VIX paths at maturity, call and put options are priced by computing the expectation of their payoffs. Implied volatility is then extracted numerically for each instrument using the model price.

In [ ]:
# Monte Carlo option valuation
# Use the previously defined strike vector
K_model = strikes_mkt / 100.0

call_prices = []
put_prices = []

for K in K_model:
    # Calculate payoffs
    payoff_c = np.maximum(VIX_T_sim - K, 0.0)
    payoff_p = np.maximum(K - VIX_T_sim, 0.0)

    # Estimate expected prices
    call_prices.append(np.mean(payoff_c))
    put_prices.append(np.mean(payoff_p))

# Invert the Black formula to obtain implied volatility
iv_model_calls = []
iv_model_puts = []

for i in range(len(K_model)):
    K = K_model[i]

    # Inversion for Calls
    vol_c = implied_vol_black(call_prices[i], F0_model, K, T, is_call=True)
    iv_model_calls.append(vol_c)

    # Inversion for Puts
    vol_p = implied_vol_black(put_prices[i], F0_model, K, T, is_call=False)
    iv_model_puts.append(vol_p)

# Convert lists to NumPy arrays
iv_model_calls = np.array(iv_model_calls)
iv_model_puts = np.array(iv_model_puts)

## 2.3 Visualisation and MSE Evaluation
The implied volatility smile produced by the one-factor Bergomi model is overlaid on market data. The mean squared error (MSE) is calculated by focusing on the Out-of-the-Money (OTM) region for greater numerical robustness (*puts* for strikes below the *forward*, and *calls* for strikes above it). The one-factor model produces a flatter smile than the one observed in the market.

In [ ]:
# Configure the figure for graphical comparison
plt.figure(figsize=(10,6))

# Plot market data
plt.plot(strikes_mkt, iv_mkt_calls, "o-", lw=2, color="blue", label="Market - Calls")
plt.plot(strikes_mkt, iv_mkt_puts, "x--", lw=2, color="blue", alpha=0.6, label="Market - Puts")

# Plot one-factor model results
plt.plot(strikes_mkt, iv_model_calls, "s-", lw=2, color="orange", label="Bergomi 1F - Calls")
plt.plot(strikes_mkt, iv_model_puts, "d--", lw=2, color="orange", alpha=0.6, label="Bergomi 1F - Puts")

# Show Forward values
plt.axvline(F0_mkt, color="blue", ls=":", alpha=0.5, label=f"F0 Market ≈ {F0_mkt:.2f}")
plt.axvline(F0_model * 100, color="orange", ls=":", alpha=0.5, label=f"F0 Model ≈ {F0_model * 100:.2f}")

plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("VIX Smile: Market vs One-Factor Bergomi Model")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Evaluate mean squared error (MSE)
erreur_globale_1f = 0.0
points_valides_1f = 0

for i in range(len(strikes_mkt)):
    K_val = strikes_mkt[i] / 100.0

    if K_val < F0_model:
        # Select Out-of-the-Money Puts
        if np.isfinite(iv_model_puts[i]) and np.isfinite(iv_mkt_puts[i]):
            erreur_globale_1f += (iv_mkt_puts[i] - iv_model_puts[i])**2
            points_valides_1f += 1
    else:
        # Select Out-of-the-Money Calls
        if np.isfinite(iv_model_calls[i]) and np.isfinite(iv_mkt_calls[i]):
            erreur_globale_1f += (iv_mkt_calls[i] - iv_model_calls[i])**2
            points_valides_1f += 1

mse_1f = erreur_globale_1f / max(points_valides_1f, 1)

print(f"\nOne-Factor Model Performance (Uncalibrated) :")
print(f"- Model VIX Future : {F0_model * 100:.4f}")
print(f"- MSE : {mse_1f:.6f}")

## 3. Mixture-of-Exponentials Model

## 3.1 Mixture-of-Exponentials Model (Simulation)
The standard Bergomi model often produces a smile that is too flat. To generate the pronounced positive skew observed in the VIX market, a mixture of exponentials is implemented. The forward variance equation incorporates a new weighting parameter $\gamma$ and two volatility-of-variance levels ($\omega_{1}$ and $\omega_{2}$).

In [ ]:
# Set initial market parameters
r = 0.0
delta = 1/12          # 30 days (VIX period)
T = T_mkt             # Valuation maturity
sigma0 = F0_mkt / 100.0
xi0 = sigma0**2       # Initial variance

# Parameters specific to the mixture of exponentials
omega1 = 5.0
omega2 = 0.5
k = 3.0
gamma = 0.5

# Monte Carlo setup
n_mc = 50000
n_rect = 120
np.random.seed(123)

# Calculate the Markovian factor X_T
var_XT = (1.0 - np.exp(-2.0 * k * T)) / (2.0 * k)

# Random generation with antithetic variates
Z = np.random.randn(n_mc // 2)
Z_anti = np.concatenate((Z, -Z))
X_T = np.sqrt(var_XT) * Z_anti

# Simulate the variance integral
du = delta / n_rect
vix2_mix = np.zeros(n_mc)

for j in range(n_rect):
    u = T + (j + 0.5) * du
    a = np.exp(-k * (u - T))

    # First mixture term: depends on X_T (integrated up to T)
    G1 = np.exp(omega1 * a * X_T - 0.5 * (omega1 * a)**2 * var_XT)

    # Second mixture term: depends on X_T and an independent increment
    #var_eps = (1.0 - a**2) / (2.0 * k)                    # Increment variance
    #eps_u   = np.sqrt(var_eps) * np.random.randn(n_mc)    # Independent noise
    #G2 = np.exp(
    #    omega2 * a * X_T + omega2 * eps_u
    #    - 0.5 * omega2**2 * (a**2 * var_XT + var_eps)
    #)
    G2 = np.exp(
        omega2 * a * X_T
        - 0.5 * (omega2 * a)**2 * var_XT
    )

    # Linear combination weighted by gamma
    xi_T_u_mix = xi0 * ((1.0 - gamma) * G1 + gamma * G2)
    vix2_mix += xi_T_u_mix * du

vix2_mix = vix2_mix / delta

# Numerical safeguard: avoid negative values under the square root
VIX_T_mix = np.sqrt(np.maximum(vix2_mix, 0.0))

# Extract the expectation for the modelled VIX Future
F0_mix = np.mean(VIX_T_mix)

## 3.2 Pricing and Extraction
Based on the paths generated by the new mixture-of-exponentials model, the options are priced. Their respective implied volatilities are then extracted numerically to reconstruct the simulated *smile*.

In [ ]:
K_model = strikes_mkt / 100.0

call_prices_mix = []
put_prices_mix = []

# Calculate expected payoffs over simulated paths
for K in K_model:
    call_prices_mix.append(np.mean(np.maximum(VIX_T_mix - K, 0.0)))
    put_prices_mix.append(np.mean(np.maximum(K - VIX_T_mix, 0.0)))

iv_model_mix_calls = []
iv_model_mix_puts = []

for i in range(len(K_model)):
    K = K_model[i]

    # Invert the Black formula for Calls
    vol_c = implied_vol_black(call_prices_mix[i], F0_mix, K, T, is_call=True)
    iv_model_mix_calls.append(vol_c)

    # Invert the Black formula for Puts
    vol_p = implied_vol_black(put_prices_mix[i], F0_mix, K, T, is_call=False)
    iv_model_mix_puts.append(vol_p)

iv_model_mix_calls = np.array(iv_model_mix_calls)
iv_model_mix_puts = np.array(iv_model_mix_puts)

## 3.3 Visualisation and Evaluation
The new *smile* generated by the mixture model is compared with the market targets. The *skew* is more pronounced, and the overall mean squared error reflects an improved fit.

In [ ]:
# Initialise the chart
plt.figure(figsize=(10,6))

# Plot market implied volatilities
plt.plot(strikes_mkt, iv_mkt_calls, "o-", lw=2, color="blue", label="Market - Calls")
plt.plot(strikes_mkt, iv_mkt_puts, "x--", lw=2, color="blue", alpha=0.6, label="Market - Puts")

# Plot mixture-model implied volatilities
plt.plot(strikes_mkt, iv_model_mix_calls, "s-", lw=2, color="green", label="Mixture Model - Calls")
plt.plot(strikes_mkt, iv_model_mix_puts, "d--", lw=2, color="green", alpha=0.6, label="Mixture Model - Puts")

# Display reference points (Futures)
plt.axvline(F0_mkt, color="blue", ls=":", alpha=0.5, label=f"F0 Market ≈ {F0_mkt:.2f}")
plt.axvline(F0_mix * 100, color="green", ls=":", alpha=0.5, label=f"F0 Model ≈ {F0_mix * 100:.2f}")

plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("VIX Smile: Market vs Mixture-of-Exponentials Model")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Calculate MSE over the Out-of-the-Money region
erreur_globale_mix = 0.0
points_valides_mix = 0

for i in range(len(strikes_mkt)):
    K_val = strikes_mkt[i] / 100.0

    if K_val < F0_mix:
        # Puts (strikes below the Forward)
        if np.isfinite(iv_model_mix_puts[i]) and np.isfinite(iv_mkt_puts[i]):
            erreur_globale_mix += (iv_mkt_puts[i] - iv_model_mix_puts[i])**2
            points_valides_mix += 1
    else:
        # Calls (strikes above the Forward)
        if np.isfinite(iv_model_mix_calls[i]) and np.isfinite(iv_mkt_calls[i]):
            erreur_globale_mix += (iv_mkt_calls[i] - iv_model_mix_calls[i])**2
            points_valides_mix += 1

mse_mix = erreur_globale_mix / max(points_valides_mix, 1)

print(f"\nMixture Model Performance (Uncalibrated) :")
print(f"- Model VIX Future : {F0_mix * 100:.4f}")
print(f"- MSE : {mse_mix:.6f}")

## 4. Model Comparison

We highlight the limitation of the one-factor Bergomi model by visually comparing market data with this model and an initial evaluation of the mixture model.

In [ ]:
# Build OTM (Out-of-the-Money) volatility curves
# Selection rule: Puts for K < F0, Calls for K >= F0
iv_mkt_otm = np.where(strikes_mkt < F0_mkt, iv_mkt_puts, iv_mkt_calls)

# Scale adjustment: convert model Futures to VIX points (×100) for the logical condition
iv_1f_otm  = np.where(strikes_mkt < F0_model * 100, iv_model_puts,     iv_model_calls)
iv_mix_otm = np.where(strikes_mkt < F0_mix   * 100, iv_model_mix_puts, iv_model_mix_calls)

# Graphical visualisation
plt.figure(figsize=(10, 6))

# Plot consolidated OTM curves
plt.plot(strikes_mkt, iv_mkt_otm, "o-", color="blue", lw=2, label="Market")
plt.plot(strikes_mkt, iv_1f_otm, "s-", color="orange", lw=2, label="One-Factor Bergomi")
plt.plot(strikes_mkt, iv_mix_otm, "d-", color="green", lw=2, label="Mixture of Exponentials")

# Visual marker for the market Future
plt.axvline(F0_mkt, color="black", ls=":", alpha=0.7, label=f"F0 Market ≈ {F0_mkt:.2f}")

plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("Comparison of VIX Smiles: Market vs Models")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Build the comparative volatility table
comparaison_otm = pd.DataFrame({
    "Strike": strikes_mkt,
    "Market": iv_mkt_otm,
    "One Factor": iv_1f_otm,
    "Mixture": iv_mix_otm
})

# Format display as percentages for presentation
comparaison_affichee = comparaison_otm.copy()
colonnes_vol = comparaison_affichee.columns[1:]
for col in colonnes_vol:
    comparaison_affichee[col] = (comparaison_affichee[col] * 100).round(2).astype(str) + " %"

# Display the summary table
print(" IMPLIED VOLATILITY COMPARISON TABLE ")
print(comparaison_affichee.to_string(index=False))
print("=====================================================\n")

# Summary of mean squared errors (MSE)
print("MODEL PERFORMANCE BEFORE CALIBRATION")
print(f"MSE - One-Factor Bergomi : {mse_1f:.6f}")
print(f"MSE - Mixture Model    : {mse_mix:.6f}")


## 5. Calibration of the Mixture-of-Exponentials Model

The objective here is to calibrate the model to VIX market implied volatility by varying the parameters $\omega_1$, $\omega_2$, $k>0$ and $\gamma\in(0,1)$. We use the Levenberg--Marquardt optimisation algorithm.

## 5.1 Calibration Preparation: Parameters and Robust Functions
Import of the optimisation modules and redefinition of the *pricing* functions with numerical bounds to ensure calibration stability. A sigmoid function is also defined to subsequently constrain the parameter $\gamma$ to the interval $(0,1)$.

In [ ]:
# Market parameters and assumptions
r = 0.0
delta = 1/12             # VIX integration period (30 days)
T = 16/365               # Maturity
sigma0 = 0.2018          # Observed initial volatility
xi0 = sigma0**2          # Initial forward variance assumed constant

# Robust analytical pricing functions
def black_call(F, K, T, vol):
    if vol <= 0 or T <= 0: return max(F - K, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return F * norm.cdf(d1) - K * norm.cdf(d2)

def black_put(F, K, T, vol):
    if vol <= 0 or T <= 0: return max(K - F, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return K * norm.cdf(-d2) - F * norm.cdf(-d1)

def implied_vol_black(price, F, K, T, is_call=True):
    intrinsic = max(F - K, 0.0) if is_call else max(K - F, 0.0)
    if price <= intrinsic + 1e-12:
        return np.nan
    upper = 10.0
    try:
        if is_call:
            return brentq(lambda vol: black_call(F, K, T, vol) - price, 1e-6, upper, maxiter=200)
        else:
            return brentq(lambda vol: black_put(F, K, T, vol) - price, 1e-6, upper, maxiter=200)
    except Exception:
        return np.nan

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

## 5.2 Bergomi Model (Mixture of Exponentials)
The model is encapsulated in a parameterised function so that it can be called iteratively by the optimisation algorithm. The initial variance curve $\xi_{0}$ is dynamically adjusted to exactly match the VIX future value observed in the market (target $F_{0}$).

In [ ]:
# Bergomi mixture-of-exponentials model, encapsulated for optimisation
def smile_model_mix(strikes_pts, F0_cible, omega1, omega2, k, gamma, n_mc=15000, n_rect=80, seed=12345):
    rng = np.random.default_rng(seed)

    var_XT = (1.0 - np.exp(-2.0 * k * T)) / (2.0 * k)
    Z = rng.standard_normal(n_mc // 2)
    Z_anti = np.concatenate((Z, -Z))
    X_T = np.sqrt(var_XT) * Z_anti

    du = delta / n_rect
    vix2_stoch = np.zeros(n_mc)

    for j in range(n_rect):
        u = T + (j + 0.5) * du
        a = np.exp(-k * (u - T))

        # Evaluate the first mixture term (dependent on X_T)
        G1 = np.exp(omega1 * a * X_T - 0.5 * (omega1**2) * (a**2) * var_XT)

        # Evaluate the second mixture term (dependent on X_T and an independent increment)
        var_eps = (1.0 - a**2) / (2.0 * k)
        eps_u   = np.sqrt(var_eps) * rng.standard_normal(n_mc)
        #G2 = np.exp(
        #    omega2 * a * X_T + omega2 * eps_u
        #    - 0.5 * omega2**2 * (a**2 * var_XT + var_eps)
        #)
        G2 = np.exp(
        omega2 * a * X_T
        - 0.5 * (omega2 * a)**2 * var_XT)

        # Integrate the purely stochastic dynamics
        stoch_part = ((1.0 - gamma) * G1 + gamma * G2)
        vix2_stoch += (stoch_part * du)

    # Dynamically adjust the initial variance level
    # Calculate the unscaled VIX
    VIX_T_unscaled = np.sqrt(np.maximum(vix2_stoch / delta, 0.0)) * 100.0

    # Calculate the adjustment ratio to reach the F0 target
    ratio_ajustement = F0_cible / np.mean(VIX_T_unscaled)

    # Apply the adjustment ratio to the paths
    VIX_T = VIX_T_unscaled * ratio_ajustement

    F0_model = np.mean(VIX_T)

    iv_model = []
    for K in strikes_pts:
        if K < F0_model:
            p_put = np.mean(np.maximum(K - VIX_T, 0.0))
            vol = implied_vol_black(p_put, F0_model, K, T, is_call=False)
        else:
            p_call = np.mean(np.maximum(VIX_T - K, 0.0))
            vol = implied_vol_black(p_call, F0_model, K, T, is_call=True)

        iv_model.append(vol)

    return np.array(iv_model), F0_model

## 5.3 Objective Definition and Calibration
The Levenberg--Marquardt algorithm is used to minimise the difference between simulated implied volatility and market implied volatility. To enforce parameter positivity and constrain $\gamma$ to $(0,1)$, mathematical transformations (logarithm and logit) are applied to the optimisation variables.

In [ ]:
# Function mapping optimised parameters
def unpack_params(theta):
    a, b, c, d = theta
    omega1 = np.exp(a)
    omega2 = np.exp(b)
    k = np.exp(c)
    gamma = sigmoid(d)
    return omega1, omega2, k, gamma

# Objective function evaluating the difference from market data
def residuals(theta):
    omega1, omega2, k, gamma = unpack_params(theta)

    # Evaluate the model with fewer paths for optimisation
    iv_model, _ = smile_model_mix(
        strikes_pts=strikes_mkt,
        F0_cible=F0_mkt,
        omega1=omega1,
        omega2=omega2,
        k=k,
        gamma=gamma,
        n_mc=15000,
        n_rect=60,
        seed=2026
    )

    # Handle undefined values
    iv_model = np.where(np.isnan(iv_model), 3.0, iv_model)

    # Return residuals calculated over the OTM region
    return iv_model - iv_mkt_otm


# Initialise starting parameters
omega1_0 = 2.0
omega2_0 = 0.2
k_0 = 3.0
gamma_0 = 0.5

# Mathematical transformations for unconstrained optimisation
log_omega1 = np.log(omega1_0)
log_omega2 = np.log(omega2_0)
log_k = np.log(k_0)
logit_gamma = np.log(gamma_0 / (1.0 - gamma_0))

# Build the initial state vector
theta0 = np.array([log_omega1, log_omega2, log_k, logit_gamma])

# Run the nonlinear least-squares algorithm
res = least_squares(
    residuals,
    theta0,
    method="lm",
    max_nfev=400,
    xtol=1e-6,
    ftol=1e-6,
    gtol=1e-6
)

# Extract calibrated parameters
theta_optimal = res.x
omega1_cal, omega2_cal, k_cal, gamma_cal = unpack_params(theta_optimal)
ratio_omega = omega1_cal / omega2_cal

print("CALIBRATION RESULTS")
print(f"omega1 optimised = {omega1_cal:.6f}")
print(f"omega2 optimised = {omega2_cal:.6f}")
print(f"k optimised      = {k_cal:.6f}")
print(f"gamma optimised  = {gamma_cal:.6f}")
print(f"Ratio w1/w2     = {ratio_omega:.3f}")

## 5.4 Validation and Visualisation
The model is run one final time with the optimised parameters using a large number of Monte Carlo paths to ensure result accuracy. The final mean squared error is measured and the calibrated *smile* is overlaid on market data. As stated in the project brief, $\gamma$ should be close to $0.5$ and the ratio $\frac{\omega_{1}}{\omega_{2}}$ close to $10$.

In [ ]:
# Final model evaluation with optimal parameters
iv_calibrated, F0_calibrated = smile_model_mix(
    strikes_pts=strikes_mkt,
    F0_cible=F0_mkt,
    omega1=omega1_cal,
    omega2=omega2_cal,
    k=k_cal,
    gamma=gamma_cal,
    n_mc=50000,
    n_rect=120,
    seed=9999
)

# Evaluate final accuracy using MSE
iv_calibrated = np.where(np.isnan(iv_calibrated), 3.0, iv_calibrated)
mse_finale = np.mean((iv_calibrated - iv_mkt_otm)**2)

print(f"Final MSE (after calibration) : {mse_finale:.6f}\n")

# Create the final comparison chart
plt.figure(figsize=(10, 6))

# Plot target market data
plt.plot(strikes_mkt, iv_mkt_otm, "o-", color="blue", lw=2, label="Market")

# Plot the curve generated by the calibrated model
plt.plot(strikes_mkt, iv_calibrated, "s-", color="red", lw=2, label="Mixture Model (Calibrated)")

# Display the VIX Future marker
plt.axvline(F0_mkt, color="black", ls=":", alpha=0.7, label=f"F0 Market ≈ {F0_mkt:.2f}")

plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("VIX Smile: Market vs Calibrated Mixture Model")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Multi-Maturity Calibration

## 6.1 Global Parameters and Core Functions
This section extends the Bergomi model to a term structure. Global market parameters are defined and the analytical *pricing* functions are initialised. The mean-reversion parameter $k$ is kept constant.

In [ ]:
# Define market assumptions and global parameters
quote_date = pd.Timestamp("2026-04-13")
sigma0 = 0.2018              # Initial volatility
xi0 = sigma0**2              # Initial forward variance
delta = 1.0 / 12.0           # Time increment (30 days)
r = 0.0                      # Risk-free interest rate

# Retrieve the previously calibrated k parameter or assign the default value
k_global = k_cal if 'k_cal' in locals() else 3.0

# Robust analytical pricing and inversion functions
def black_call(F, K, T, vol):
    if vol <= 0 or T <= 0: return max(F - K, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return F * norm.cdf(d1) - K * norm.cdf(d2)

def black_put(F, K, T, vol):
    if vol <= 0 or T <= 0: return max(K - F, 0.0)
    srt = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol * T) / srt
    d2 = d1 - srt
    return K * norm.cdf(-d2) - F * norm.cdf(-d1)

def implied_vol_black(price, F, K, T, is_call=True):
    intrinsic = max(F - K, 0.0) if is_call else max(K - F, 0.0)
    if price <= intrinsic + 1e-12:
        return np.nan
    try:
        if is_call:
            return brentq(lambda vol: black_call(F, K, T, vol) - price, 1e-6, 10.0, maxiter=200)
        else:
            return brentq(lambda vol: black_put(F, K, T, vol) - price, 1e-6, 10.0, maxiter=200)
    except:
        return np.nan

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

## 6.2 Multi-Maturity Simulation Engine
The mixture-of-exponentials model is adapted to accept maturity $T$ as a dynamic input variable. The function simulates VIX paths for a given maturity, adjusts the initial variance level to match the *Future* target, and then extracts Out-of-the-Money (OTM) implied volatilities.

In [ ]:
# Bergomi mixture-model simulation function parameterised by maturity
def smile_model_mix_multi(strikes_pts, F0_cible, T_val, omega1, omega2, k_val, gamma_val, n_mc=30000, n_rect=80, seed=12345):
    rng = np.random.default_rng(seed)

    var_XT = (1.0 - np.exp(-2.0 * k_val * T_val)) / (2.0 * k_val)

    Z = rng.standard_normal(n_mc // 2)
    Z_anti = np.concatenate((Z, -Z))
    X_T = np.sqrt(var_XT) * Z_anti

    du = delta / n_rect
    vix2_stoch = np.zeros(n_mc)

    for j in range(n_rect):
        u = T_val + (j + 0.5) * du
        a = np.exp(-k_val * (u - T_val))

        # Evaluate the first mixture term
        G1 = np.exp(omega1 * a * X_T - 0.5 * (omega1**2) * (a**2) * var_XT)

        # Evaluate the second mixture term
        var_eps = (1.0 - a**2) / (2.0 * k_val)
        eps_u   = np.sqrt(var_eps) * rng.standard_normal(n_mc)
        #G2 = np.exp(
         #   omega2 * a * X_T + omega2 * eps_u
         #   - 0.5 * omega2**2 * (a**2 * var_XT + var_eps)
        #)
        G2 = np.exp(
        omega2 * a * X_T
        - 0.5 * (omega2 * a)**2 * var_XT)

        stoch_part = ((1.0 - gamma_val) * G1 + gamma_val * G2)
        vix2_stoch += (stoch_part * du)

    # Dynamically adjust the curve level to reach the F0 target
    VIX_T_unscaled = np.sqrt(np.maximum(vix2_stoch / delta, 0.0)) * 100.0
    ratio_ajustement = F0_cible / np.mean(VIX_T_unscaled)
    VIX_T = VIX_T_unscaled * ratio_ajustement

    F0_model = np.mean(VIX_T)

    iv_model = []
    for K in strikes_pts:
        if K < F0_model:
            p_put = np.mean(np.maximum(K - VIX_T, 0.0))
            vol = implied_vol_black(p_put, F0_model, K, T_val, is_call=False)
        else:
            p_call = np.mean(np.maximum(VIX_T - K, 0.0))
            vol = implied_vol_black(p_call, F0_model, K, T_val, is_call=True)
        iv_model.append(vol)

    return np.array(iv_model), F0_model

## 6.3 Market Data Processing (Term Structure)
The entire quote file is processed to extract OTM implied volatility *smiles* for all available liquid maturities. Outlier or illiquid observations are filtered out.

In [ ]:
# Load the full dataset
df_all = pd.read_csv("vix_quotedata (2).csv", skiprows=3)

# Extract unique maturities
toutes_maturites = df_all["Expiration Date"].unique()
expiries_valides = []
data_par_maturite = {}

for exp in toutes_maturites:
    df_mat = df_all[df_all["Expiration Date"] == exp].copy()

    cols = ["Strike", "Bid", "Ask", "Bid.1", "Ask.1"]
    for col in cols:
        df_mat[col] = pd.to_numeric(df_mat[col], errors="coerce")

    # Apply the liquidity filter
    df_mat = df_mat[(df_mat["Bid"] > 0) & (df_mat["Ask"] > 0) & (df_mat["Bid.1"] > 0) & (df_mat["Ask.1"] > 0)].copy()

    # Restrict the strike range for analysis
    df_mat = df_mat[(df_mat["Strike"] >= 15) & (df_mat["Strike"] <= 30)].copy()

    if len(df_mat) >= 5:
        # Calculate reference prices and the implied Forward
        df_mat["Mid_Call"] = 0.5 * (df_mat["Bid"] + df_mat["Ask"])
        df_mat["Mid_Put"] = 0.5 * (df_mat["Bid.1"] + df_mat["Ask.1"])
        df_mat["Synthetic_Forward"] = df_mat["Mid_Call"] - df_mat["Mid_Put"] + df_mat["Strike"]
        F0_mkt_exp = df_mat["Synthetic_Forward"].median()

        # Convert maturity to a year fraction
        jours = (pd.to_datetime(exp) - quote_date).days
        T_exp = max(jours, 1) / 365.0

        # Extract implied volatilities
        iv_calls, iv_puts = [], []
        for _, row in df_mat.iterrows():
            K = row["Strike"]
            iv_calls.append(implied_vol_black(row["Mid_Call"], F0_mkt_exp, K, T_exp, is_call=True))
            iv_puts.append(implied_vol_black(row["Mid_Put"], F0_mkt_exp, K, T_exp, is_call=False))

        df_mat["IV_Recalc_Call"] = iv_calls
        df_mat["IV_Recalc_Put"] = iv_puts
        df_mat = df_mat.dropna(subset=["IV_Recalc_Call", "IV_Recalc_Put"]).sort_values("Strike")

        if len(df_mat) >= 5:
            expiries_valides.append(exp)
            # Build the OTM curve
            strikes = df_mat["Strike"].values
            iv_mkt_otm = np.where(strikes < F0_mkt_exp, df_mat["IV_Recalc_Put"].values, df_mat["IV_Recalc_Call"].values)

            data_par_maturite[exp] = {
                "T": T_exp,
                "F0_mkt": F0_mkt_exp,
                "strikes": strikes,
                "iv_mkt_otm": iv_mkt_otm
            }
            print(f"{exp} retained : {len(strikes)} valid strikes (F0 Market = {F0_mkt_exp:.2f})")

## 6.4 Sequential Calibration by Maturity
A loop iterates over all valid maturities. For each maturity slice, the nonlinear least-squares algorithm optimises the parameters $\omega_{1}$, $\omega_{2}$ and $\gamma$ to minimise the error relative to the market while keeping $k$ constant.

In [ ]:
results = []
fits = {}

# Iterate over each maturity for local calibration
for exp in expiries_valides:
    mkt_info = data_par_maturite[exp]
    T_exp = mkt_info["T"]
    F0_mkt_exp = mkt_info["F0_mkt"]
    strikes_mkt = mkt_info["strikes"]
    iv_mkt_otm = mkt_info["iv_mkt_otm"]

    # Define the cost function for the optimiser
    def residuals(theta):
        omega1 = np.exp(theta[0])
        omega2 = np.exp(theta[1])
        gamma = sigmoid(theta[2])

        iv_model, _ = smile_model_mix_multi(
            strikes_pts=strikes_mkt,
            F0_cible=F0_mkt_exp,
            T_val=T_exp,
            omega1=omega1,
            omega2=omega2,
            k_val=k_global,
            gamma_val=gamma,
            n_mc=15000,
            n_rect=60,
            seed=2026
        )

        iv_model = np.where(np.isnan(iv_model), 3.0, iv_model)
        return iv_model - iv_mkt_otm

    # Initialise starting parameters with mathematical transformations
    omega1_0, omega2_0, gamma_0 = 4.0, 0.3, 0.5
    theta0 = [np.log(omega1_0), np.log(omega2_0), np.log(gamma_0 / (1.0 - gamma_0))]

    # Run the optimisation algorithm
    res = least_squares(residuals, x0=theta0, method="lm", xtol=1e-5, ftol=1e-5, max_nfev=100)

    # Back-transform the optimal parameters
    omega1_opt = np.exp(res.x[0])
    omega2_opt = np.exp(res.x[1])
    gamma_opt = sigmoid(res.x[2])

    # High-precision evaluation with calibrated parameters
    iv_fit, F0_fit = smile_model_mix_multi(
        strikes_pts=strikes_mkt,
        F0_cible=F0_mkt_exp,
        T_val=T_exp,
        omega1=omega1_opt,
        omega2=omega2_opt,
        k_val=k_global,
        gamma_val=gamma_opt,
        n_mc=50000,
        n_rect=120,
        seed=9999
    )

    mse = np.mean((iv_fit - iv_mkt_otm)**2)

    # Store results
    results.append({
        "Maturity": exp,
        "T (years)": np.round(T_exp, 4),
        "omega1": omega1_opt,
        "omega2": omega2_opt,
        "k (fixed)": k_global,
        "gamma": gamma_opt,
        "Ratio w1/w2": omega1_opt / omega2_opt,
        "F0 Model": F0_fit,
        "F0 Market": F0_mkt_exp,
        "MSE": mse
    })

    fits[exp] = {
        "strikes": strikes_mkt,
        "iv_mkt": iv_mkt_otm,
        "iv_model": iv_fit
    }

## 6.5 Summary and Visualisation of the Calibrated Volatility Surface
A summary table of the calibrated parameters for each maturity is displayed. A grid of charts is then generated to visually compare the dynamics of the smiles produced by the model with actual market quotes across the full term structure.

In [ ]:
# Display the calibration summary table
print("CALIBRATION SUMMARY TABLE")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Configure the grid display
n_plots = len(expiries_valides)
ncols = 2
nrows = int(np.ceil(n_plots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

# Iteratively plot charts for each maturity
for idx, exp in enumerate(expiries_valides):
    ax = axes[idx]
    plot_data = fits[exp]

    ax.plot(plot_data["strikes"], plot_data["iv_mkt"], "o-", color="blue", lw=2, label="Market")
    ax.plot(plot_data["strikes"], plot_data["iv_model"], "s--", color="red", lw=2, label="Calibrated Mixture")

    ax.axvline(data_par_maturite[exp]["F0_mkt"], color="gray", ls=":", label="F0 Market")

    ax.set_title(f"Maturity : {exp}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Strike")
    ax.set_ylabel("Implied volatility")
    ax.grid(True, alpha=0.3)
    ax.legend()

# Hide unused axes
for j in range(n_plots, len(axes)):
    axes[j].axis("off")

plt.suptitle("Multi-Maturity Calibration of the VIX Smile", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()